
### **1. Data loading and text preprocessing**


1. Download the dataset from HuggingFace - PL-Guard

In [ ]:
from huggingface_hub import notebook_login
import pandas as pd
import numpy as np
import ast

notebook_login()

In [ ]:
from datasets import load_dataset

dataset = load_dataset("NASK-PIB/PL-Guard", 'test')
dataset_adv = load_dataset("NASK-PIB/PL-Guard", 'test_adversarial')

After downloading the dataset, we can explore its structure and contents.

In [ ]:
print(dataset.column_names)
print(dataset_adv.column_names)

In [ ]:
split = dataset['test']
sentences = split['text']
categories = split['category']

print("Split name:", split)
print("Rows:", len(split))
print("Columns:", split.column_names)
print("Features:", split.features)

df_head = split.select(range(min(5, len(split)))).to_pandas()
display(df_head)

texts = split["text"]
cats = split["category"]

n_empty = sum((t is None) or (not str(t).strip()) for t in texts)
print("Empty/None texts:", n_empty)

categories_count = pd.Series(cats).value_counts()
categories_share = (categories_count / categories_count.sum()).round(4)

display(pd.DataFrame({"count": categories_count, "share": categories_share}))


### **2. Text representation - using embeddings**


Computer doesn't understand words. It understands numbers. Therefore, we need to convert the text data into numerical format using embeddings.
What is embedding? - Embedding is a way to represent words or phrases as vectors of numbers in a continuous vector space.

This allows us to capture semantic relationships between words based on their meanings and contexts.
To generate embeddings for our text data, we leverage the Sentence-BERT (SBERT) architecture. It is better suited for generating sentence-level embeddings compared to traditional BERT models, as SBERT was specifically designed so that the cosine distance between vectors reflects semantic similarity.

Specifically, we implement the `paraphrase-multilingual-MiniLM-L12-v2` model from the SentenceTransformers library—a smaller and faster version of SBERT that maintains high performance.

In [ ]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np

model_name = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'  #unique name of the model from HuggingFace
model = SentenceTransformer(
    model_name)  #download the model, create an instance of SentenceTransformer class (wraps complex functionality like tokenization, data flow into simple function), load the model


Tokenization is the process of breaking down text into smaller units called tokens (words, subwords, or characters, depending on the strategy). The tokens are then mapped to integer IDs using the model’s vocabulary. These IDs are converted into vectors via an embedding layer, and then passed through the Transformer blocks (self-attention, feed-forward/linear layers, and normalizations) to produce contextualized token embeddings. Next, a pooling operation (e.g., mean/max pooling, typically masking out padding tokens) aggregates token embeddings into a fixed-size sentence embedding, often followed by L2-normalization to keep embeddings on a consistent scale for similarity computations.

In [ ]:
print(model.get_sentence_embedding_dimension())
print(model)


In this model embeddings after pooling aren't normalized by default (norm isn't equal 1.0). Every sentence from this model will be converted into a vector of dimension 384. Normalization can be enabled via encode(normalize_embeddings=True)

In [ ]:
embeddings = model.encode(sentences, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True)

In [ ]:
adv_split = dataset_adv["test_adversarial"]
df_adv = adv_split.to_pandas()

emb_org = embeddings
emb_adv = model.encode(df_adv['text'].tolist(), convert_to_numpy=True, normalize_embeddings=True,
                       show_progress_bar=True)

In [ ]:
print(f'Shape of embeddings: {embeddings.shape}')
print(f'Example embedding vector: {embeddings[:5]}')

Embeddings shape is (900,384) that means we have 900 sentences and each sentence is represented by a vector of 384 dimensions. If two sentences are semantically similar, their embeddings will be close in the vector space.


### 3. Dimensionality reduction and visualization


Our embeddings are in 384-dimensional space. To visualize them, we need to reduce their dimensionality to 2D or 3D. We can use techniques like PCA (Principal Component Analysis), t-SNE (t-Distributed Stochastic Neighbor Embedding) or UMAP (Uniform Manifold Approximation and Projection) for this purpose. First, we will use PCA to reduce the embeddings to 2D.

#### 3.1 PCA dimensionality reduction

In [ ]:
from sklearn.decomposition import PCA

pca_2d = PCA(n_components=2)
embeddings_2d = pca_2d.fit_transform(embeddings)

pca_3d = PCA(n_components=3)
embeddings_3d = pca_3d.fit_transform(embeddings)

print(f'2D: {sum(pca_2d.explained_variance_ratio_) * 100:.2f}% variance explained by 2 components)')
print(f'3D: {sum(pca_3d.explained_variance_ratio_) * 100:.2f}% variance explained by 3 components)')

PCA is a linear projection and in our embeddings the first 2 components explain ~14% variance (3 components ~20%), which is typical for high-dimensional sentence embeddings. Therefore, 2D/3D PCA plots are useful mainly for visualization/sanity-check, and we will also try non-linear methods (UMAP/t-SNE) that can better preserve local neighborhood structure

In [ ]:
import plotly.express as px

fig = px.scatter(
    x=embeddings_2d[:, 0],  # first PCA component [take all rows, first column]
    y=embeddings_2d[:, 1],  # second PCA component
    color=categories,
    hover_name=sentences,
    title='PCA 2D of Sentence Embeddings',
    labels={'x': 'PCA Component 1', 'y': 'PCA Component 2'},
)

fig.update_traces(marker=dict(size=8, opacity=0.7))
fig.update_layout(legend_title_text='Kategorie złośliwości')
fig.show()

#### 3.2 t-SNE dimensionality reduction

Now we'll check t-SNE for dimensionality reduction.

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, perplexity=30, random_state=42,
            max_iter=1000)  #perplexity is related to the number of nearest neighbors that point takes into consideration
embeddings_tsne = tsne.fit_transform(embeddings)

fig_tsne = px.scatter(
    x=embeddings_tsne[:, 0],
    y=embeddings_tsne[:, 1],
    color=categories,
    hover_name=sentences,
    title='t-SNE 2D of Sentence Embeddings',
    labels={'x': 't-SNE Component 1', 'y': 't-SNE Component 2'},
)
fig_tsne.update_traces(marker=dict(size=8, opacity=0.7))
fig_tsne.show()

In [ ]:
pca_50_model = PCA(n_components=50, random_state=42)
embeddings_pca_50 = pca_50_model.fit_transform(embeddings)

tsne_with_pca_50 = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=1000)
embeddings_tsne_with_pca_50 = tsne_with_pca_50.fit_transform(embeddings_pca_50)

fig_tsne_with_pca_50 = px.scatter(
    x=embeddings_tsne_with_pca_50[:, 0],
    y=embeddings_tsne_with_pca_50[:, 1],
    color=categories,
    hover_name=sentences,
    title='t-SNE 2D of Sentence Embeddings (PCA to 50D first)',
    labels={'x': 't-SNE Component 1', 'y': 't-SNE Component 2'},
)
fig_tsne_with_pca_50.show()

In [ ]:
embeddings_pca_150 = PCA(n_components=150, random_state=42).fit_transform(embeddings)
tsne_with_pca_150 = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=1000)
embeddings_tsne_with_pca_150 = tsne_with_pca_150.fit_transform(embeddings_pca_150)

fig_tsne_150 = px.scatter(
    x=embeddings_tsne_with_pca_150[:, 0],
    y=embeddings_tsne_with_pca_150[:, 1],
    color=categories,
    hover_name=sentences,
    title='t-SNE 2D of Sentence Embeddings (PCA to 150D first)',
    labels={'x': 't-SNE Component 1', 'y': 't-SNE Component 2'},
)
fig_tsne_150.show()

Trustworthiness measures how well local neighbourhoods (k-nearest neighbours) from the original space are preserved after dimensionality reduction. The score ranges from 0 to 1, where values closer to 1 mean better preservation of local structure.

Because we use pipelines such as PCA → t-SNE, we report trustworthiness in two ways. First, end-to-end (384D → 2D): we compare the final 2D map against the original 384D embedding space. This reflects the combined distortion introduced by all steps (e.g., PCA plus t-SNE). Second, t-SNE-only (input space → 2D): we compare the final 2D map against the actual input space of t-SNE (e.g., PCA-50 or PCA-150). This isolates the distortion introduced by t-SNE alone, given its input.

In [ ]:
from sklearn.manifold import trustworthiness


def T(orig, red, k):
    return trustworthiness(orig, red, n_neighbors=k)


rows = []
for k in [5, 10, 20]:
    rows.append({
        "k": k,

        # t-SNE directly (end-to-end 384 -> 2D)
        "tSNE_384→2D (end-to-end)": T(embeddings, embeddings_tsne, k),

        # PCA50 -> t-SNE (end-to-end and t-SNE-only)
        "PCA50→tSNE (end-to-end 384→2D)": T(embeddings, embeddings_tsne_with_pca_50, k),
        "PCA50→tSNE (tSNE-only 50→2D)": T(embeddings_pca_50, embeddings_tsne_with_pca_50, k),

        # PCA150 -> t-SNE (end-to-end and t-SNE-only)
        "PCA150→tSNE (end-to-end 384→2D)": T(embeddings, embeddings_tsne_with_pca_150, k),
        "PCA150→tSNE (tSNE-only 150→2D)": T(embeddings_pca_150, embeddings_tsne_with_pca_150, k),
    })

df_T = pd.DataFrame(rows)
display(df_T.round(4))


We report trustworthiness for two evaluation settings: end-to-end (384D→2D), which measures how well the 2D map preserves neighbourhoods from the original 384D embedding space, and tSNE-only (PCA-space→2D), which measures preservation with respect to the PCA-reduced space. These two settings are not directly comparable because the reference space differs.

In the end-to-end (384D→2D) comparison, applying PCA before t-SNE improves neighbourhood preservation over running t-SNE directly on 384D embeddings. The baseline tSNE_384→2D scores 0.9532 / 0.9293 / 0.9045, while PCA50→tSNE (end-to-end) reaches 0.9646 / 0.9419 / 0.9189, which is the best end-to-end result for all tested k. PCA150→tSNE (end-to-end) yields a smaller gain (0.9559 / 0.9327 / 0.9070).

In the tSNE-only (PCA-space→2D) setting, trustworthiness is higher overall. PCA50→tSNE (tSNE-only) achieves the highest values in the table (0.9737 / 0.9563 / 0.9359), indicating that t-SNE preserves local neighbourhoods very well relative to the 50D PCA space.

Overall, PCA to 50 dimensions appears to be a practical “sweet spot”: it provides the strongest end-to-end neighbourhood preservation (relative to the original 384D embeddings) and also yields excellent stability in the PCA-space mapping. This supports our later choice to primarily work with PCA-50D features for downstream visualisation and analysis.

In [ ]:
from sklearn.preprocessing import normalize

X_cluster = normalize(embeddings_pca_50)
X_plot = embeddings_tsne_with_pca_50

#### 3.3 UMAP dimensionality reduction

Now we'll check UMAP for dimensionality reduction.

In [ ]:
import umap

umap_model = umap.UMAP(n_components=2, n_neighbors=5, min_dist=0.1,
                       random_state=42)  #n_neighbors controls local vs global structure, min_dist controls spacing between points
embeddings_umap = umap_model.fit_transform(embeddings)

fig_umap = px.scatter(
    x=embeddings_umap[:, 0],
    y=embeddings_umap[:, 1],
    color=categories,
    hover_name=sentences,
    title='UMAP 2D of Sentence Embeddings',
    labels={'x': 'UMAP Component 1', 'y': 'UMAP Component 2'},
)
fig_umap.show()

In [ ]:
def calculate_trustworthiness(original_embeddings, reduced_embeddings, n_neighbors):
    return trustworthiness(original_embeddings, reduced_embeddings, n_neighbors=n_neighbors)


for k in [5, 10, 20]:
    trust_umap = calculate_trustworthiness(embeddings, embeddings_umap, n_neighbors=k)
    print(f'Trustworthiness of UMAP 2D: {trust_umap:.4f}')

We evaluated neighborhood preservation using trustworthiness (k=5/10/20). For UMAP, the best configuration was n_neighbors=15 (0.9481/0.9326/0.9076). However, t-SNE preceded by PCA to 50D achieved higher trustworthiness (0.9646/0.9419/0.9189), indicating better preservation of local neighborhoods in our setting.

The best dimensionality reduction method for our sentence embeddings appears to be t-SNE applied after reducing dimensions to 50D with PCA. This combination achieves the highest trustworthiness scores across multiple neighborhood sizes (k=5,10,20), indicating it best preserves local structures compared to other methods tested (PCA alone, t-SNE directly on 384D, UMAP).


### **4. Clustering**

#### 4.1 K-Means Clustering

This algorithm partitions the data into K clusters by minimizing the variance within each cluster. It iteratively assigns points to the nearest cluster centroid and updates centroids based on the mean of assigned points.

But how we should choose the optimal number of clusters (K)? One common method is the "elbow method," which involves plotting the inertia (sum of squared distances from each point to its assigned cluster center) against different values of K. The idea is to look for an "elbow" point in the plot where the rate of decrease in inertia sharply changes, indicating that adding more clusters beyond this point yields diminishing returns in terms of reducing inertia.

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

inertia = []
inertia_full = []
k_range = range(1, 21)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cluster)
    inertia.append(km.inertia_)

for k in k_range:
    km_full = KMeans(n_clusters=k, random_state=42, n_init=10)
    km_full.fit(embeddings)
    inertia_full.append(km_full.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(k_range, inertia, 'bx-')
plt.xlabel('Num of clusters (k)')
plt.ylabel('Inertia (Sum of squared distances)')
plt.title('The elbow method for K-Means')
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(k_range, inertia_full, 'bx-')
plt.xlabel('Num of clusters (k)')
plt.ylabel('Inertia (Sum of squared distances)')
plt.title('The elbow method for K-Means (full embeddings 384D)')
plt.show()


In [ ]:
from sklearn.metrics import silhouette_score

scores = []
for k in range(2, 21):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_cluster)
    scores.append(silhouette_score(X_cluster, labels))

print(scores)

The silhouette score measures how well each sample fits within its assigned cluster compared to other clusters. For a point i, it compares (a) the average distance to points in its own cluster (cohesion) with (b) the average distance to points in the nearest neighboring cluster (separation). The score ranges from −1 to 1: values close to 1 indicate well-separated clusters, values near 0 indicate overlapping clusters, and negative values suggest that many points may be assigned to the wrong cluster.

The elbow method does not provide a clear, unambiguous choice of k in our case: the inertia decreases smoothly and the “flattening” point can be interpreted in different ways (e.g., only around k ≈ 3,6, 11,15). In addition, the silhouette scores obtained for K-Means are consistently low across the tested range of k (approximately 0.06–0.1), which suggests that the data do not form well-separated, compact clusters under the assumptions of K-Means. Therefore, instead of selecting k based on an uncertain elbow and weak silhouette values, we set k = 15 to match the number of dataset categories and to enable a direct comparison between the unsupervised clusters and the ground-truth labels.

For completeness we compute elbow curves for both spaces, but the subsequent clustering experiments are performed on the PCA-50 + normalized embeddings, motivated by our earlier results indicating that PCA-50 provides a cleaner, less noisy representation.

In [ ]:
kmeans_15 = KMeans(n_clusters=15, random_state=42, n_init=10)
labels_kmeans_15 = kmeans_15.fit_predict(X_cluster)

In [ ]:
df_visualisation = pd.DataFrame({
    'tsne_x': embeddings_tsne_with_pca_50[:, 0],
    'tsne_y': embeddings_tsne_with_pca_50[:, 1],
    'category': categories,
    'kmeans_15': labels_kmeans_15.astype(str),
    "text": sentences
})

print(df_visualisation.shape)
print(df_visualisation.head())

For plotting the clustering results, we embed the data into 2D using t-SNE computed on the PCA-50 representation (embeddings_tsne_with_pca_50). This choice is motivated by our earlier neighbourhood-preservation evaluation: among the tested 2D projections, PCA-50 → t-SNE provided the most reliable visual structure (best trustworthiness in the end-to-end setting and consistently strong scores across k). Importantly, t-SNE is used only for visualisation here; the actual K-Means clustering is performed in the PCA-50 (normalized) space (X_cluster), and the 2D map is used to inspect how cluster assignments relate to the ground-truth categories.

In [ ]:
fig = px.scatter(
    df_visualisation,
    x='tsne_x',
    y='tsne_y',
    color='kmeans_15',
    symbol='category',  #true labels from PL-Guard dataset
    hover_data=['text', 'category', 'kmeans_15'],
    title='K-Means (k=15) clustering on PCA-50 embeddings, visualized with t-SNE (2D)',
    labels={'tsne_x': 't-SNE Component 1', 'tsne_y': 't-SNE Component 2'},
)
fig.show()

In [ ]:
fig = px.scatter(
    df_visualisation,
    x='tsne_x', y='tsne_y',
    color='category',
    hover_data=['text', 'kmeans_15'],
    title='t-SNE colored by true category (hover shows KMeans cluster)'
)
fig.show()


In [ ]:
fig = px.scatter(
    df_visualisation,
    x='tsne_x', y='tsne_y',
    color='kmeans_15',
    hover_data=['text', 'category'],
    title='t-SNE colored by KMeans cluster (hover shows true category)'
)
fig.show()


In [ ]:
df_visualisation['safety'] = df_visualisation['category'].apply(lambda c: 'safe' if c == 'safe' else 'unsafe')

fig = px.scatter(
    df_visualisation,
    x='tsne_x', y='tsne_y',
    color='kmeans_15',
    symbol='safety',
    hover_data=['text', 'category', 'safety'],
    title='t-SNE: color=KMeans cluster, symbol=safe/unsafe'
)
fig.show()


In [ ]:
ct_km = pd.crosstab(df_visualisation['kmeans_15'], df_visualisation['category'])
purity_km = ct_km.max(axis=1).sum() / ct_km.values.sum()
print("purity (kmeans):", purity_km)
print(ct_km)


Using K-Means with k = 15, the clustering shows limited alignment with the ground-truth categories. The purity is ~0.571, meaning that on average only about 57% of samples in each cluster belong to the cluster’s dominant class.

This is consistent with the low silhouette scores observed earlier (≈ 0.05–0.07 across different k), suggesting that the embeddings do not form strongly separated, spherical clusters in the Euclidean sense assumed by K-Means.

Overall, K-Means provides only weakly separable clusters on this embedding space; clusters tend to mix multiple categories rather than cleanly recovering the dataset labels.

#### 4.2 Hierarchical Clustering

In [ ]:
from sklearn.cluster import AgglomerativeClustering

agglo = AgglomerativeClustering(n_clusters=15, linkage='ward')
labels_agglo = agglo.fit_predict(X_cluster)

df_visualisation['agglo_15'] = labels_agglo.astype(str)

In [ ]:
fig = px.scatter(
    df_visualisation,
    x='tsne_x',
    y='tsne_y',
    color='agglo_15',
    hover_data=['text', 'category', 'agglo_15'],
    title='t-SNE (2D) visualization colored by Agglomerative clusters (k=15, trained on PCA-50 embeddings)',
    labels={'tsne_x': 't-SNE Component 1', 'tsne_y': 't-SNE Component 2'},
)
fig.show()

In [ ]:
import pandas as pd

ct = pd.crosstab(df_visualisation['agglo_15'], df_visualisation['category'])
purity = ct.max(axis=1).sum() / ct.values.sum()
print("purity:", purity)
print(ct)


In [ ]:
from sklearn.metrics import silhouette_score

print(silhouette_score(X_cluster, labels_agglo))


Agglomerative clustering with Ward linkage and k = 15 yields very similar behavior. The purity is ~0.538, slightly lower than K-Means but only by a marginal amount.

The silhouette score is ~0.0786, which is also very low, indicating poor separation between clusters and substantial overlap in the embedding space.

In summary, Agglomerative (Ward) does not substantially improve clustering quality over K-Means for these embeddings; both methods produce clusters that only partially correspond to the labeled categories.

#### 4.3 DBSCAN Clustering

In [ ]:
from sklearn.cluster import DBSCAN

dbscan = DBSCAN(eps=0.5, min_samples=5,  #default values
                metric='euclidean')  #eps - how close points should be to be considered part of the same neighborhood, min_samples - minimum number of points to form a dense region
labels_dbscan = dbscan.fit_predict(X_cluster)

df_visualisation['dbscan'] = labels_dbscan.astype(str)

n_outliers = sum(labels_dbscan == -1)
print(f'Number of outliers detected by DBSCAN: {n_outliers}')

In [ ]:
df_visualisation['is_outlier'] = (df_visualisation['dbscan'] == '-1')

fig = px.scatter(
    df_visualisation,
    x='tsne_x', y='tsne_y',
    color='dbscan',  # DBSCAN clusters
    symbol='is_outlier',
    hover_data=['text', 'category', 'dbscan', 'is_outlier'],
    title='t-SNE (2D) visualization colored by DBSCAN labels (fit on PCA-50 embeddings; outliers highlighted)',
    labels={'tsne_x': 't-SNE Component 1', 'tsne_y': 't-SNE Component 2'}
)
fig.show()


In [ ]:
labels = labels_dbscan
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
print("clusters:", n_clusters, "outliers:", n_outliers)


In [ ]:
for eps in [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8]:
    for ms in [3, 5, 7, 9]:
        labels = DBSCAN(eps=eps, min_samples=ms, metric='euclidean').fit_predict(X_cluster)
        n_out = (labels == -1).sum()
        n_cl = len(set(labels)) - (1 if -1 in labels else 0)
        print(f"eps={eps:>4}, min_samples={ms}: clusters={n_cl:>2}, outliers={n_out}")


In [ ]:
def plot_dbscan_on_tsne(df_visualisation, X_for_dbscan, eps, min_samples, title_suffix=""):
    labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(X_for_dbscan)

    dfp = df_visualisation.copy()
    dfp["dbscan_label"] = labels
    dfp["is_outlier"] = (labels == -1)
    dfp["dbscan_label_str"] = dfp["dbscan_label"].astype(str)

    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_outliers = int((labels == -1).sum())
    n_points = len(labels)

    fig = px.scatter(
        dfp,
        x="tsne_x", y="tsne_y",
        color="dbscan_label_str",
        symbol="is_outlier",
        hover_data=["text", "category", "dbscan_label_str", "is_outlier"],
        title=(
            f"t-SNE (2D) colored by DBSCAN labels "
            f"(eps={eps}, min_samples={min_samples}) {title_suffix} | "
            f"clusters={n_clusters}, outliers={n_outliers}/{n_points}"
        )
    )
    fig.show()

    print(f"DBSCAN(eps={eps}, min_samples={min_samples}) -> clusters={n_clusters}, outliers={n_outliers}/{n_points}")
    print("Cluster sizes (label -> count):")
    print(dfp["dbscan_label"].value_counts().sort_index())


# X_cluster = embeddings_pca_50
plot_dbscan_on_tsne(df_visualisation, X_cluster, eps=0.75, min_samples=3)
plot_dbscan_on_tsne(df_visualisation, X_cluster, eps=0.80, min_samples=5)


For eps=0.75, DBSCAN detects several clusters and a moderate number of outliers (noise). Outliers appear mainly at the edges of structures in the t-SNE projection, which is consistent with intuition: these are points with lower neighborhood density in PCA-50 space. This variant preserves greater cluster “resolution” (less merging into one group) at the expense of a higher percentage of noise.

Increasing eps to 0.80 causes more points to be “absorbed” by clusters: the number of outliers decreases, but at the same time, some previously separate structures may be merged into larger clusters. This shows a typical trade-off in DBSCAN: higher eps → less noise, but a higher risk of merging different groups into a single label.

In [ ]:
from sklearn.neighbors import NearestNeighbors

X = X_cluster  # embeddings_pca_50
k = 3  # min_samples

nn = NearestNeighbors(n_neighbors=k)
nn.fit(X)
distances, _ = nn.kneighbors(X)

k_dist = np.sort(distances[:, -1])

plt.figure()
plt.plot(k_dist)
plt.title(f"k-distance plot (k={k}) on X_cluster (PCA-50 + L2 norm)")
plt.xlabel("Points sorted by distance")
plt.ylabel(f"Distance to {k}-th nearest neighbor")

for eps in [0.75, 0.80]:
    plt.axhline(eps, linestyle="--")

plt.show()

k = 5
nn = NearestNeighbors(n_neighbors=k)
nn.fit(X)
distances, _ = nn.kneighbors(X)

k_dist = np.sort(distances[:, -1])

plt.figure()
plt.plot(k_dist)
plt.title(f"k-distance plot (k={k}) on X_cluster (PCA-50 + L2 norm)")
plt.xlabel("Points sorted by distance")
plt.ylabel(f"Distance to {k}-th nearest neighbor")

for eps in [0.75, 0.80]:
    plt.axhline(eps, linestyle="--")

plt.show()



1. The k-distance curves for k=3 and k=5 indicate a relatively sparse data structure, which is characteristic of PCA-50 data with L2 normalization. No sharp elbow: The curves do not exhibit a distinct "elbow" at low values. High baseline: Instead of starting near zero, the curves begin at relatively high distance values (approximately 0.38 for k=3 and approximately 0.55 for k=5). Curve shape: The plots rise smoothly and only begin to level off noticeably around the 0.75–0.80 range on the y-axis. This suggests that neighborhood radii smaller than this threshold are too restrictive to capture the majority of data points.

2. The grid search results show that the previously considered eps range (0.55–0.70) is ineffective for this dataset. At eps=0.55, nearly the entire dataset is classified as noise (approximately 95%, 855 out of 900 points). Even at eps=0.70 with min_samples=3, a large fraction of points remains classified as outliers (approximately 68%, 615 out of 900 points). These high outlier counts indicate that DBSCAN detects only small, fragmented micro-clusters in this range rather than coherent global structure.

3. More meaningful cluster behavior emerges only when eps reaches the 0.75–0.80 range. For min_samples=3, the number of clusters increases up to eps=0.70 (47 clusters) and then decreases at eps=0.75–0.80 (38 to 32 clusters). This reduction in cluster count, together with a substantial decrease in outliers (from 615 down to 409), indicates the onset of cluster merging. For stricter density requirements (min_samples = 5, 7, 9), valid clusters do not appear until eps is close to or above 0.70.

4. The analysis shows that the effective eps range is substantially higher than initially estimated. Recommended investigation range: eps approximately 0.75–0.85. A practical starting point is eps=0.80, which offers a reasonable trade-off between cluster connectivity and noise, reducing the outlier fraction to roughly 45%. Eps values below 0.75 lead to excessive fragmentation and unstable clustering results.


#### 4.4 HDBSCAN Clustering

In [ ]:
import hdbscan
import numpy as np

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=20,
    min_samples=2,  # as in DBSCAN; more = more outliers
    metric='euclidean'
)

labels_hdb = clusterer.fit_predict(X_cluster)

n_outliers = np.sum(labels_hdb == -1)
n_clusters = len(set(labels_hdb)) - (1 if -1 in labels_hdb else 0)

print("HDBSCAN clusters:", n_clusters)
print("HDBSCAN outliers:", n_outliers)


In [ ]:
for mcs in [5, 10, 15, 20, 30]:
    for ms in [None, 1, 2, 5, 10]:
        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=mcs,
            min_samples=ms,
            metric='euclidean'  # albo 'euclidean' jeśli embeddings są L2=1
        )
        labels = clusterer.fit_predict(X_cluster)
        n_out = np.sum(labels == -1)
        n_cl = len(set(labels)) - (1 if -1 in labels else 0)
        print(f"mcs={mcs:>2}, ms={str(ms):>4} -> clusters={n_cl:>2}, outliers={n_out}")

HDBSCAN (Hierarchical DBSCAN) is a density-based clustering method that can discover clusters of varying density and explicitly label points that do not belong to any dense region as noise (outliers, label = -1). Two key hyperparameters are: min_cluster_size, which sets the minimum size of a cluster, and min_samples, which controls how conservative the algorithm is when deciding whether a point is part of a dense region (larger values typically produce more outliers). In our grid search we observed the expected trade-off: configurations that yield a larger number of clusters also tend to label more points as outliers. For example, with min_cluster_size=5 the setting min_samples=1 produced 28 clusters and 385 outliers, while increasing min_samples to 2 resulted in 25 clusters but a higher outlier count (445), reflecting a stricter density requirement for core points. A more conservative choice, e.g. min_cluster_size=5 with min_samples=None (defaulting to min_samples=min_cluster_size), reduced the structure to 3 clusters and 293 outliers, illustrating how larger effective density thresholds merge clusters and change the noise assignment.

In [ ]:
configs = [
    ("hdb_mcs5_ms2", 5, 2),
    ("hdb_mcs5_ms1", 5, 1),
    ("hdb_mcs10_ms1", 10, 1),
]

hover_cols = [c for c in ["category", "text"] if c in df_visualisation.columns]

for col_name, mcs, ms in configs:
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=mcs,
        min_samples=ms,
        metric="euclidean"
    )
    labels = clusterer.fit_predict(X_cluster)

    n_outliers = int(np.sum(labels == -1))
    n_clusters = int(len(set(labels)) - (1 if -1 in labels else 0))
    print(f"{col_name}: clusters={n_clusters}, outliers={n_outliers}")

    df_visualisation[col_name] = labels.astype(str)
    out_col = f"{col_name}_is_outlier"
    df_visualisation[out_col] = (labels == -1)

    fig = px.scatter(
        df_visualisation,
        x="tsne_x", y="tsne_y",
        color=col_name,
        symbol=out_col,
        hover_data=hover_cols,
        title=f"t-SNE (2D) visualization colored by HDBSCAN labels (fit on PCA-50), mcs={mcs}, ms={ms}",
        labels={"tsne_x": "t-SNE Component 1", "tsne_y": "t-SNE Component 2"}
    )
    fig.show()


#### 5. Outlier detection

We treat points labeled as noise (-1) by density-based methods as outliers. We report the overall outlier rate, compare outlier rates across categories, and inspect example prompts to understand what types of inputs fall into low-density regions of the embedding space. DBSCAN outliers depend strongly on eps, therefore we compare two reasonable eps values (0.75 and 0.80) and use HDBSCAN as a robustness check.

In [ ]:
def report_outliers(df, outlier_mask, method_name, n_examples=10, random_state=42):
    outlier_mask = np.asarray(outlier_mask).astype(bool)
    n = len(df)
    n_outliers = int(outlier_mask.sum())
    print(f"{method_name} outliers: {n_outliers}/{n} ({n_outliers / n:.2%})")

    rates = (
        pd.DataFrame({"category": df["category"], "is_outlier": outlier_mask})
        .groupby("category")["is_outlier"]
        .mean()
        .sort_values(ascending=False)
    )
    display(rates.to_frame("outlier_rate").round(4))

    df_examples = df.loc[outlier_mask, ["category", "text"]].copy()
    if len(df_examples) > 0:
        display(df_examples.sample(min(n_examples, len(df_examples)), random_state=random_state))
    else:
        print("No outliers found.")


5.1 DBSCAN outliers

In [ ]:
dbscan_configs = [
    ("DBSCAN eps=0.75, min_samples=3", 0.75, 3),
    ("DBSCAN eps=0.80, min_samples=3", 0.80, 3),
]

for name, eps, ms in dbscan_configs:
    labels = DBSCAN(eps=eps, min_samples=ms, metric="euclidean").fit_predict(X_cluster)
    out_mask = (labels == -1)
    report_outliers(df_visualisation, out_mask, name)


For eps=0.75 and min_samples=3, DBSCAN labels 517/900 points as outliers (57.44%), so the setting is very strict (sparse density in the embedding space). Outlier rates are highest for several unsafe subcategories, most notably unsafe\S7 (0.80), unsafe\S2 (0.72) and unsafe\S6 (0.72), while even the safe class has a relatively high outlier rate (0.53). The lowest outlier rates are still substantial (e.g., unsafe\S9 = 0.40, unsafe\S13 = 0.42), indicating that “outlierness” is widespread across categories rather than concentrated in a few rare classes. The sampled outliers include both clearly harmful prompts and apparently “benign”/refusal-like or informational ones, suggesting that DBSCAN is primarily capturing local density/rarity in embedding space, not a monotonic “toxicity” scale.


Increasing eps to 0.80 makes DBSCAN more permissive, reducing outliers to 409/900 (45.44%) — still a large fraction, but noticeably lower than for eps=0.75. Category-level outlier rates decrease across the board (e.g., safe: 0.53 → 0.41, unsafe\S7: 0.80 → 0.60, unsafe\S3: 0.68 → 0.56), but the pattern remains similar: the highest outlier rates are still observed for unsafe\S2 (0.72), unsafe\S6 (0.66), and unsafe\S7 (0.60). The lowest categories (e.g., unsafe\S13 = 0.26, unsafe\S12 = 0.30) no longer look extreme, which supports the expected trade-off: a larger eps absorbs more borderline points into clusters (fewer outliers), but at the cost of looser density criteria, which can merge semantically heterogeneous regions.

5.2 HDBSCAN outliers

In [ ]:
for out_col in ["hdb_mcs5_ms2_is_outlier", "hdb_mcs5_ms1_is_outlier", "hdb_mcs10_ms1_is_outlier"]:
    if out_col in df_visualisation.columns:
        report_outliers(df_visualisation, df_visualisation[out_col].values, f"HDBSCAN {out_col}")


HDBSCAN hdb_mcs5_ms2_is_outlier — outliers: 445/900 (49.44%)
This configuration labels almost half of the dataset as outliers, so it is highly strict in terms of density requirements.
Outlier rates are elevated across all categories (roughly 0.34–0.72), not concentrated in a few classes. The highest rates are observed for unsafe\S3 (0.72) and unsafe\S11 (0.70), followed by unsafe\S6 (0.62) and unsafe\S7 (0.58). Even safe has a high outlier rate (0.475).
Interpretation: with these parameters, HDBSCAN treats the embedding space as having relatively few stable dense regions; “outlierness” reflects local density/rarity, not simply whether a prompt is safe vs unsafe.


HDBSCAN hdb_mcs5_ms1_is_outlier — outliers: 385/900 (42.78%)
This setting is more permissive than `mcs=5, ms=2`, reducing outliers by about 6.7 percentage points (49.44% → 42.78%).
The category pattern remains similar, but rates drop in most classes (e.g., unsafe\S11: 0.70 → 0.54, safe: 0.475 → 0.425). The highest outlier rates are still in unsafe subcategories such as unsafe\S3 (0.64) and unsafe\S6 (0.60), while the lowest classes are still far from zero (e.g., unsafe\S12 = 0.26, unsafe\S8 = 0.28).
Practical takeaway: lowering `min_samples` makes it easier for points to be considered part of a cluster, but the method still marks a large portion of the dataset as noise—suggesting that the embedding geometry is relatively sparse or fragmented at this scale.

HDBSCAN hdb_mcs10_ms1_is_outlier — outliers: 414/900 (46.00%)
With a larger `min_cluster_size` (10), the outlier fraction increases again to 46.00%, i.e., it becomes stricter than `mcs=5, ms=1` but slightly less strict than `mcs=5, ms=2`.
The outlier-rate profile is again broad across categories: the highest rates appear for unsafe\S3 (0.70) and unsafe\S6 (0.66), while safe remains high (0.47). Several unsafe categories stay around 0.40–0.54, and the lowest observed is still unsafe\S12 = 0.30.
Interpretation: increasing `min_cluster_size` makes clusters harder to form/maintain, pushing more borderline points into noise. Overall, outlier detection here is clearly parameter-sensitive, but all three configurations agree that a substantial fraction of samples lies outside dense regions in embedding space.


In [ ]:
labels_db_075 = DBSCAN(eps=0.75, min_samples=3, metric="euclidean").fit_predict(X_cluster)
out_db_075 = (labels_db_075 == -1)
out_hdb = (labels_hdb == -1)

inter = int(np.sum(out_db_075 & out_hdb))
union = int(np.sum(out_db_075 | out_hdb))
jacc = inter / union if union > 0 else 0.0

print(f"Outlier overlap (DBSCAN 0.75/3 vs HDBSCAN):")
print(f"Intersection: {inter}, Union: {union}, Jaccard: {jacc:.3f}")


The overlap is high (Jaccard = 0.741), meaning DBSCAN(0.75, 3) and the chosen HDBSCAN definition mark largely the same points as outliers. Specifically, the intersection contains 437 samples, while the union contains 590 samples. Relative to DBSCAN’s 517 outliers, 80 are flagged only by DBSCAN, whereas HDBSCAN adds 73 additional outliers (total 510). Interpretation: despite differences in how DBSCAN and HDBSCAN model density, at this parameterization they converge on a consistent notion of “low-density / atypical” points in the embedding space.

Across DBSCAN and HDBSCAN, the outlier fraction is generally high (≈ 42%–57%) for the settings shown here (e.g., DBSCAN eps=0.75: 57.44%, eps=0.80: 45.44%; HDBSCAN configs: 42.78%–49.44%). This indicates strong density fragmentation in the embedding space: many samples lie outside stable dense regions. Category-level outlier rates also do not map cleanly to *safe vs unsafe*—even safe can have a substantial outlier rate—supporting the view that “outlierness” reflects embedding-space rarity (style/topic/formulation), not simply toxicity.


5.3 Prompts far from the rest

In [ ]:
E = X_cluster

centroid = E.mean(axis=0)
centroid = centroid / (np.linalg.norm(centroid) + 1e-12)

cos_sim = E @ centroid
dist_to_centroid = 1.0 - cos_sim

q = 0.95  #upper quantil
thr = np.quantile(dist_to_centroid, q)
out_mask = dist_to_centroid >= thr

df_visualisation["dist_to_centroid"] = dist_to_centroid

print(f"Distance-to-centroid threshold (Q{int(q * 100)}) = {thr:.4f}")
report_outliers(df_visualisation, out_mask, f"Distance-based (cosine) outliers, Q{int(q * 100)}")

display(
    df_visualisation.loc[out_mask, ["category", "dist_to_centroid", "text"]]
    .sort_values("dist_to_centroid", ascending=False)
    .head(15)
)

fig = px.histogram(
    df_visualisation,
    x="dist_to_centroid",
    nbins=40,
    title="Distance-to-centroid (cosine) distribution",
    labels={"dist_to_centroid": "Cosine distance to centroid"}
)
fig.add_vline(x=thr, line_dash="dash", annotation_text=f"Q{int(q * 100)}")
fig.show()


DBSCAN/HDBSCAN identify outliers as low-density points (local anomalies), while our approach detects global outliers: prompts that lie farthest from the embedding centroid.

Using cosine distance to the centroid in the X_cluster space (PCA-50 + L2 normalization), we set the outlier cut-off at the 95th percentile (Q95 = 1.4314). This means that the outliers are the 5% most globally atypical prompts in terms of overall semantics, yielding 45/900 (5.00%) observations classified as outliers. The histogram shows a clear right-tail, and the Q95 line isolates exactly this tail.

Importantly, the outliers are highly class-dependent, but not in the way “more harmful = more outlying”. The safe class has by far the largest outlier share (outlier_rate = 0.205), while only two unsafe subcategories show non-zero outlier rates (unsafe\nS11 = 0.040 and unsafe\nS3 = 0.040); all remaining unsafe categories are 0.000 in this run. This suggests that “safe” prompts (often short, templated refusal-style responses such as “Nie mogę pomóc w…”) form a semantically distinct mode relative to the typical corpus, pushing them far from the global centroid.

Example outliers are dominated by these safe refusal/meta statements, with only occasional unsafe instances. Therefore, the outlier label here should be interpreted as semantic atypicality with respect to the entire dataset, not as a direct indicator of harmfulness. Additionally, the top distance-to-centroid table shows near-duplicate refusal phrases among the most distant points, indicating that global outliers can also be driven by a separate, stylistically consistent “refusal cluster” rather than uniquely rare topics.


5.4 Adversarial_move outliers

In [ ]:
from sklearn.metrics.pairwise import cosine_distances

move_cos = np.diag(cosine_distances(emb_org, emb_adv))  #cosine distance
move_euc = np.linalg.norm(emb_org - emb_adv, axis=1)

df_adv["move_cosine"] = move_cos
df_adv["move_euc"] = move_euc

In [ ]:
def report_move_outliers(df_adv, out_mask, method_name, n_examples=10):
    out_mask = np.asarray(out_mask).astype(bool)
    n = len(df_adv)
    n_out = int(out_mask.sum())
    print(f"{method_name} outliers: {n_out}/{n} ({n_out / n:.2%})")

    rates = (
        pd.DataFrame({"category": df_adv["category"], "is_outlier": out_mask})
        .groupby("category")["is_outlier"]
        .mean()
        .sort_values(ascending=False)
    )
    display(rates.to_frame("outlier_rate").round(4))

    cols = [c for c in ["category", "num_changes", "modification_types", "move_cosine", "original_text", "text"] if
            c in df_adv.columns]
    display(
        df_adv.loc[out_mask, cols]
        .sort_values("move_cosine", ascending=False)
        .head(n_examples)
    )


q = 0.95
thr = df_adv["move_cosine"].quantile(q)
out_mask = df_adv["move_cosine"] >= thr

print(f"Move-cosine threshold (Q{int(q * 100)}) = {thr:.4f}")
report_move_outliers(df_adv, out_mask, f"Adversarial move-outliers (move_cosine), Q{int(q * 100)}", n_examples=12)

fig = px.histogram(
    df_adv,
    x="move_cosine",
    nbins=40,
    title="Distribution of adversarial embedding shift (move_cosine) with outlier threshold",
    labels={"move_cosine": "Cosine distance (original vs adversarial)"}
)
fig.add_vline(x=thr, line_dash="dash", annotation_text=f"Q{int(q * 100)}")
fig.show()


We analyzed pairs outliers (original → adversarial) that cause an unusually large embedding shift measured by cosine distance. The Q95 threshold was 0.6374, which resulted in 45/900 (5%) observations with the largest shift. The histogram shows a strongly skewed distribution: most modifications result in small shifts (close to 0), but there is a distinct tail up to values close to 1, indicating cases where the text after perturbation becomes semantically “far” from the original. Movement outliers are relatively more frequent in the safe class (0.14) than in the individual unsafe subclasses (max. approx. 0.08), suggesting that for “safe” (often short, template refusals), aggressive form perturbations (OCR/keyboard/diacritics/swap) may change the representation more.

#### 4.7 Impact of adversarial changes

We evaluate how adversarial edits shift each sentence in the embedding space. Since dataset['test']['text'] is identical to dataset_adv['test_adversarial']['original_text'], we can reuse the already computed embeddings as the “original” representation and compute embeddings only for the adversarial texts.

Cosine distance (move_cosine) – measures the change in direction of the embedding vectors (semantic shift). With normalized embeddings, cosine distance is the most natural measure because it focuses on semantic orientation rather than vector magnitude.

Euclidean distance (move_euc) – measures the straight-line distance between embeddings in the 384-dimensional space. With normalized embeddings, it is strongly related to cosine distance but still offers an intuitive “how far did the point move” interpretation.

In [ ]:
from sklearn.metrics.pairwise import cosine_distances

move_cos = np.diag(cosine_distances(emb_org, emb_adv))  #cosine distance
move_euc = np.linalg.norm(emb_org - emb_adv, axis=1)

df_adv["move_cosine"] = move_cos
df_adv["move_euc"] = move_euc

print(df_adv[["move_cosine", "move_euc"]].describe())

For 900 samples, the average cosine distance is 0.178, with median 0.089. This suggests that most adversarial edits cause a small-to-moderate semantic shift: half of the examples move by less than ~0.09 in cosine distance, while the upper quartile reaches ~0.235, indicating a noticeable change for a substantial subset.

The Euclidean distance shows a similar pattern (mean 0.498, median 0.423). Because embeddings are normalized, Euclidean distance largely reflects the same phenomenon as cosine distance (larger angular differences lead to larger Euclidean moves). The maximum observed shift is high (cosine ≈ 1.02, Euclidean ≈ 1.43), meaning that for a few samples the adversarial modifications substantially change the embedding and may significantly alter neighborhood relations (which can affect clustering assignments and classifier decisions).

Overall, the distribution indicates that adversarial perturbations do not uniformly “break” representations: many samples remain close to their original embedding, but there exists a tail of examples with large shifts, which are the most likely to impact downstream tasks.

In [ ]:
q25, q50, q75 = df_adv["move_cosine"].quantile([0.25, 0.50, 0.75])

fig = px.histogram(df_adv, x="move_cosine", nbins=40,
                   title="Distribution of adversarial embedding shift (cosine distance)")
fig.add_vline(x=q25, line_dash="dash", annotation_text="25%")
fig.add_vline(x=q50, line_dash="dash", annotation_text="median")
fig.add_vline(x=q75, line_dash="dash", annotation_text="75%")
fig.show()

Most samples have small shift (median ~0.089), but the tail reaches >0.23 for 25% of points.

In [ ]:
def to_list(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)  # np. "['delete','swap']" -> ['delete','swap']
        except:
            return [x]
    return []


df_adv["mod_types_list"] = df_adv["modification_types"].apply(to_list)

# 2) Zrób "long format": jedna linia = jeden typ modyfikacji dla danego sample
df_long = df_adv.explode("mod_types_list").rename(columns={"mod_types_list": "mod_type"})
df_long["mod_type"] = df_long["mod_type"].astype(str)

# 3) Top 6 najczęstszych typów
top6 = df_long["mod_type"].value_counts().head(6).index
df_top = df_long[df_long["mod_type"].isin(top6)].copy()

# 4) Boxplot
fig = px.box(
    df_top,
    x="mod_type",
    y="move_cosine",
    points="outliers",
    title="Cosine shift by modification type (top 6 most frequent)",
    labels={"mod_type": "Modification type", "move_cosine": "Cosine distance"},
)
fig.show()

The boxplot compares how different adversarial modification types affect the embedding shift. Higher medians and wider spreads indicate that a given modification type typically causes larger and more variable changes in sentence representations

In [ ]:
grp = df_adv.groupby("num_changes", as_index=False)["move_cosine"].agg(["mean", "median", "count"]).reset_index()

fig = px.line(grp, x="num_changes", y="median",
              title="Adversarial shift vs number of changes (median cosine distance)")
fig.update_layout(xaxis_title="Number of changes", yaxis_title="Median cosine distance")
fig.show()


More changes generally lead to larger semantic shift (median increases with num_changes)

In [ ]:
top = df_adv.sort_values("move_cosine", ascending=False).head(20)

fig = px.bar(top, x=top.index.astype(str), y="move_cosine",
             hover_data=["category", "num_changes", "modification_types"],
             title="Top 20 largest adversarial shifts (cosine distance)")
fig.update_layout(xaxis_title="Sample", yaxis_title="Cosine distance")
fig.show()


This plot ranks the 20 samples with the largest embedding shift caused by adversarial edits. Each bar corresponds to one sentence; the y-axis shows cosine distance between the original and adversarial embeddings. The hover tooltip provides attack metadata: the ground-truth category, the number of applied edits (num_changes), and the list of edit types (modification_types). Values close to 1.0 indicate a very strong semantic shift in the embedding space.

In [ ]:
cols = ["move_cosine", "num_changes", "category", "original_text", "text"]
top5_view = df_adv.sort_values("move_cosine", ascending=False).head(5)[cols]
top5_view


These are the five prompts with the highest cosine distance between the original and adversarial embeddings. A high cosine distance indicates a strong change in the direction of the embedding vector in 384D space, i.e., the model represents the modified text as much less semantically similar to the original (in terms of angle).

4.7.3 Does it break clustering?

We test whether adversarial edits “break” clustering by measuring cluster assignment stability under a fixed clustering model. We keep the original K-Means model (k=15) trained on X_cluster (PCA-50 + L2 normalization) and project the adversarial embeddings into the same PCA-50 space (using the PCA fit on the original embeddings), followed by L2 normalization. Then we compute the switch rate: the fraction of samples for which the adversarial version is assigned to a different cluster than the original.
Next, we break down switch rates by category, by number of changes (num_changes, binned), and by the most frequent modification types. Finally, we compare move_cosine distributions for switched vs non-switched samples to verify that large embedding shifts are associated with cluster instability.

In [ ]:
from sklearn.preprocessing import normalize
import ast

# ------------------------------------------------------------
# 3.7.3 Does it break clustering? (KMeans switch rate analysis)
# Assumes you already have:
# - embeddings (original 384D, normalized)
# - emb_adv (adversarial 384D, normalized)
# - kmeans_15 (trained on X_cluster)
# - labels_kmeans_15 (cluster labels for original samples)
# - df_adv with columns: category, num_changes, modification_types, move_cosine
# ------------------------------------------------------------

# 2) Project adversarial embeddings into PCA-50 and apply the same L2 normalization
X_adv_cluster = normalize(pca_50_model.transform(emb_adv))

# 3) Compare cluster assignments: original vs adversarial (same KMeans model)
labels_orig = labels_kmeans_15
labels_adv = kmeans_15.predict(X_adv_cluster)

df_adv["cluster_switched"] = (labels_adv != labels_orig)

# --- Overall switch rate ---
switch_rate = df_adv["cluster_switched"].mean()
print(f"Overall switch rate (KMeans k=15): {switch_rate:.2%} "
      f"({df_adv['cluster_switched'].sum()}/{len(df_adv)})")

# 4) Breakdown A: switch rate by category
print("\nSwitch rate by category:")
display(
    df_adv.groupby("category", observed=True)["cluster_switched"]
    .mean()
    .sort_values(ascending=False)
    .to_frame("switch_rate")
    .round(4)
)

# 5) Breakdown B: switch rate by number of changes (binned)
bins = [0, 2, 5, 10, 20, 100]
bin_labels = ["1-2", "3-5", "6-10", "11-20", "21+"]
df_adv["num_changes_bin"] = pd.cut(df_adv["num_changes"], bins=bins, labels=bin_labels, include_lowest=True)

print("\nSwitch rate by num_changes (binned):")
display(
    df_adv.groupby("num_changes_bin", observed=True)["cluster_switched"]
    .mean()
    .to_frame("switch_rate")
    .round(4)
)


# 6) Breakdown C: switch rate by modification type (top 6 most frequent)
# Helper: parse the list stored in modification_types
def parse_mod_types(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)  # handles strings like "['ocr', 'swap']"
        except:
            return [x]
    return []


df_adv["mod_type"] = df_adv["modification_types"].apply(parse_mod_types)
df_long = df_adv.explode("mod_type").dropna(subset=["mod_type"])
df_long["mod_type"] = df_long["mod_type"].astype(str)

top_k = 6
top_types = df_long["mod_type"].value_counts().head(top_k).index

print(f"\nSwitch rate by modification type (top {top_k} types):")
display(
    df_long[df_long["mod_type"].isin(top_types)]
    .groupby("mod_type", observed=True)["cluster_switched"]
    .mean()
    .sort_values(ascending=False)
    .to_frame("switch_rate")
    .round(4)
)

# 7) Sanity check: do switched samples have larger embedding shifts (move_cosine)?
print("\nmove_cosine summary (switch vs no-switch):")
display(
    df_adv.groupby("cluster_switched", observed=True)["move_cosine"]
    .describe()[["count", "mean", "50%", "75%", "max"]]
    .rename(index={False: "no_switch", True: "switch"})
    .round(4)
)


1) Overall switch rate: Adversarial edits change the KMeans cluster assignment for **21.56%** of samples (**194/900**). This means that roughly **one in five** prompts moves to a different region of the clustering space after perturbation, so clustering is not fully stable under adversarial noise.

2) Switch rate by category: Switching is **not uniform across classes**. The highest switch rates appear for **unsafe\nS2 (0.36)**, **safe (0.335)** and **unsafe\nS1 (0.32)**, suggesting these categories are more sensitive to text perturbations. The lowest rate is for **unsafe\nS14 (0.04)**, indicating much higher cluster stability for that category.

3) Switch rate by num_changes (binned): There is a clear trend: **more changes → more switching**. With only **1–2** edits the switch rate is very low (**~2.6%**), but it increases to **~18% (3–5)**, **~22% (6–10)** and **~25% (11–20)**. This supports the intuition that stronger perturbations destabilize the embedding-based clustering.

4) Switch rate by modification type + move_cosine summary: Among the most frequent modification types, **swap, OCR and keyboard** yield the highest switch rates (≈ **0.25–0.27**), while insert/diacritic are slightly lower but still substantial. The `move_cosine` summary confirms the mechanism: switched samples have a much larger embedding shift (**mean ~0.39, median ~0.35**) than non-switched ones (**mean ~0.12, median ~0.067**), so cluster changes are strongly associated with large representation moves.


### 4.8 Classification

4.8.1

We train supervised classifiers to predict the prompt category using the embedding vectors as features. We split the original dataset into train/test with stratification and evaluate three baselines: Logistic Regression, Linear SVM and Random Forest. For each model we report accuracy, macro precision, macro recall and the confusion matrix on the original test split.
To test robustness, we evaluate the same trained models on the adversarial versions of the test samples (same labels, modified text). We then report the performance drop between original and adversarial test. Finally, we test a simple robustness improvement by augmenting the training set with a subset of adversarial samples and re-evaluating both on original and adversarial test sets.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# X,y for original
X = embeddings
y_text = df_visualisation["category"].values

le = LabelEncoder()
y = le.fit_transform(y_text)

# keep indices to align later if needed
idx = np.arange(len(y))

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, idx, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", len(y_train), "Test size:", len(y_test))
print("Num classes:", len(le.classes_))


4.8.2

We use a Pipeline to combine preprocessing and the classifier into a single reproducible workflow. This ensures that all preprocessing steps (e.g., scaling) are fitted only on the training split and then applied unchanged to the test/adversarial data, preventing data leakage.
For linear models such as Logistic Regression and Linear SVM we include a StandardScaler, which standardizes each embedding dimension to zero mean and unit variance. Even when embeddings are L2-normalized, individual dimensions can still have different variances; scaling often improves numerical stability and makes regularization/margins behave consistently across features.
Tree-based models (e.g., Random Forest) are much less sensitive to feature scale, so scaling is not required there.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report


def eval_model(name, classifier, Xtr, ytr, Xte, yte):
    classifier.fit(Xtr, ytr)
    prediction = classifier.predict(Xte)

    accuracy = accuracy_score(yte, prediction)
    precision, recall, f1score, _ = precision_recall_fscore_support(yte, prediction, average="macro",
                                                                    zero_division=0)  #macro: precision/recall/F1 separately for each class, then calculate the simple average across classes (each class has the same weight)

    print(f"\n{name}")
    print(
        f"Accuracy: {accuracy:.4f} | Macro Precision: {precision:.4f} | Macro Recall: {recall:.4f} | Macro F1: {f1score:.4f}")
    print("Confusion matrix:")
    display(pd.DataFrame(confusion_matrix(yte, prediction)))

    return {"model": name, "accuracy": accuracy, "precision": precision, "recall": recall, "f1score": f1score,
            "classifier": classifier}


models = [
    ("LogReg", Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=2000, n_jobs=None))
    ])),
    ("LinearSVM", Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LinearSVC())
    ])),
    ("RandomForest", RandomForestClassifier(n_estimators=300, random_state=42))
]

results_orig = []
trained = {}

for name, classifier in models:
    out = eval_model(name, classifier, X_train, y_train, X_test, y_test)
    results_orig.append({k: out[k] for k in ["model", "accuracy", "precision", "recall", "f1score"]})
    trained[name] = out["classifier"]

display(pd.DataFrame(results_orig).sort_values("f1score", ascending=False))


We also report the confusion matrix to analyze error patterns beyond aggregate metrics. Rows correspond to true labels and columns to predicted labels; diagonal entries are correct classifications while off-diagonal entries indicate misclassifications. This allows us to identify which categories are most frequently confused and whether errors concentrate between semantically similar unsafe subcategories. Such analysis is useful for interpreting the model and for understanding which classes are most affected by adversarial perturbations

On the original test split, Logistic Regression achieves the best overall performance with Accuracy = 0.706 and Macro F1 = 0.667, making it the strongest baseline among the tested models. Random Forest has slightly lower Macro F1 = 0.653, despite having the highest macro precision (0.730), which indicates it is more conservative (fewer false positives) but misses more true cases (lower recall 0.635). Linear SVM performs worst overall (Macro F1 = 0.638), mainly due to lower macro precision (0.648). Since we report macro-averaged metrics, each class contributes equally, so the scores reflect performance across all categories rather than being dominated by the largest classes.

4.8.3

In [ ]:
X_adv_test = emb_adv[idx_test]
y_adv_test = y_test  # same labels, only text is modified

results_adv = []

for name, classifier in trained.items():
    prediction_adv = classifier.predict(X_adv_test)

    accuracy_adv = accuracy_score(y_adv_test, prediction_adv)
    precision_adv, recall_adv, f1score_adv, _ = precision_recall_fscore_support(
        y_adv_test, prediction_adv, average="macro", zero_division=0
    )

    print(f"\n{name} (Adversarial test)")
    print(
        f"Accuracy: {accuracy_adv:.4f} | Macro Precision: {precision_adv:.4f} | Macro Recall: {recall_adv:.4f} | Macro F1: {f1score_adv:.4f}")
    print("Confusion matrix:")
    display(pd.DataFrame(confusion_matrix(y_adv_test, prediction_adv)))

    results_adv.append({
        "model": name,
        "accuracy_adv": accuracy_adv,
        "precision_adv": precision_adv,
        "recall_adv": recall_adv,
        "f1score_adv": f1score_adv
    })

df_adv_scores = pd.DataFrame(results_adv)
display(df_adv_scores.sort_values("f1score_adv", ascending=False))

# Compare original vs adversarial
df_orig_scores = pd.DataFrame(results_orig)
df_compare = df_orig_scores.merge(df_adv_scores, on="model")

df_compare["accuracy_drop"] = df_compare["accuracy"] - df_compare["accuracy_adv"]
df_compare["f1score_drop"] = df_compare["f1score"] - df_compare["f1score_adv"]

display(df_compare.sort_values("f1score_drop", ascending=False))


The first table reports only adversarial test metrics. The second table merges original vs adversarial results and adds drop columns

We evaluate the trained classifiers on the adversarial versions of the same test samples (labels unchanged). All models degrade compared to the original test set, confirming that adversarial perturbations reduce classification robustness. On adversarial data, RandomForest achieves the best performance (Accuracy 0.622, Macro F1 0.587), followed closely by LogReg (Accuracy 0.611, Macro F1 0.580), while LinearSVM performs worst (Macro F1 0.528). The comparison table additionally reports the performance drop: LinearSVM is the most sensitive (largest F1 drop ≈ 0.110), whereas RandomForest shows the smallest drop (≈ 0.065), indicating higher robustness to perturbations.

4.8.4

In [ ]:
# 3.8.4 Adversarial augmentation (train-time robustness)
# Idea: add a subset of adversarial TRAIN samples to the training set,
# then re-train models and evaluate on (1) original test and (2) adversarial test.

adv_fraction = 0.30  # fraction of TRAIN samples to add in adversarial form
random_seed = 42
rng = np.random.RandomState(random_seed)

# Indices of samples that are in the training split (referring to the original dataset order)
train_indices = idx_train

# How many adversarial training samples we add
n_adv_added = int(len(train_indices) * adv_fraction)

# Randomly pick which TRAIN samples will be added as adversarial examples
adv_added_indices = rng.choice(train_indices, size=n_adv_added, replace=False)

# Build the augmented training set:
# - original training embeddings (X_train)
# - plus adversarial embeddings for selected TRAIN indices (emb_adv[adv_added_indices])
X_train_augmented = np.vstack([X_train, emb_adv[adv_added_indices]])

# Labels for augmented set:
# - original training labels (y_train)
# - plus labels of those selected samples (same labels, because only text changed)
y_train_augmented = np.concatenate([y_train, y[adv_added_indices]])

print(f"Augmented train size: {len(y_train_augmented)} (added {n_adv_added} adversarial samples)")

# Adversarial test set corresponds to the SAME test indices, but using adversarial embeddings
X_test_adversarial = emb_adv[idx_test]
y_test_adversarial = y_test


# Re-create fresh models (important: do NOT reuse already-fitted models)
def build_models_fresh():
    return [
        ("LogReg", Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(max_iter=2000))
        ])),
        ("LinearSVM", Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", LinearSVC(max_iter=20000))
        ])),
        ("RandomForest", RandomForestClassifier(n_estimators=300, random_state=random_seed))
    ]


augmented_results = []

for model_name, model in build_models_fresh():
    # Train on augmented training set
    model.fit(X_train_augmented, y_train_augmented)

    # Evaluate on ORIGINAL test
    pred_test_orig = model.predict(X_test)
    acc_orig = accuracy_score(y_test, pred_test_orig)
    prec_orig, rec_orig, f1_orig, _ = precision_recall_fscore_support(
        y_test, pred_test_orig, average="macro", zero_division=0
    )

    # Evaluate on ADVERSARIAL test
    pred_test_adv = model.predict(X_test_adversarial)
    acc_adv = accuracy_score(y_test_adversarial, pred_test_adv)
    prec_adv, rec_adv, f1_adv, _ = precision_recall_fscore_support(
        y_test_adversarial, pred_test_adv, average="macro", zero_division=0
    )

    augmented_results.append({
        "model": model_name,
        "accuracy_test_orig": acc_orig,
        "macro_f1_test_orig": f1_orig,
        "accuracy_test_adv": acc_adv,
        "macro_f1_test_adv": f1_adv
    })

df_aug = pd.DataFrame(augmented_results)

print("\nAugmented training results (sorted by adversarial Macro F1):")
display(df_aug.sort_values("macro_f1_test_adv", ascending=False))


We augment the training set by adding adversarial versions of 30% of the training samples (+216 examples). After augmentation, LogReg achieves the best overall performance both on the clean test (Accuracy 0.717, Macro F1 0.674) and on the adversarial test (Accuracy 0.633, Macro F1 0.599). RandomForest remains competitive but performs worse on adversarial data (Macro F1 0.576), while LinearSVM shows the lowest adversarial Macro F1 (0.561). Overall, this simple augmentation improves robustness (higher adversarial performance) while keeping clean-test performance at a similar level for the best model.

In [ ]:
import pandas as pd

# ------------------------------------------------------------
# Compare baseline (4.8.3) vs augmented training (4.8.4)
# baseline table: df_compare
# augmented table: df_aug
# We create a clean, consistent schema and compute deltas.
# ------------------------------------------------------------

# 1) BASELINE (4.8.3): trained on clean train, evaluated on clean+adversarial test
baseline = df_compare[[
    "model",
    "accuracy", "f1score",
    "accuracy_adv", "f1score_adv"
]].copy()

baseline = baseline.rename(columns={
    "accuracy": "clean_acc_before",
    "f1score": "clean_f1_before",
    "accuracy_adv": "adv_acc_before",
    "f1score_adv": "adv_f1_before",
})

# 2) AUGMENTED (4.8.4): trained on augmented train, evaluated on clean+adversarial test
augmented = df_aug[[
    "model",
    "accuracy_test_orig", "macro_f1_test_orig",
    "accuracy_test_adv", "macro_f1_test_adv"
]].copy()

augmented = augmented.rename(columns={
    "accuracy_test_orig": "clean_acc_after",
    "macro_f1_test_orig": "clean_f1_after",
    "accuracy_test_adv": "adv_acc_after",
    "macro_f1_test_adv": "adv_f1_after",
})

# 3) MERGE + DELTAS (after - before)
comparison = baseline.merge(augmented, on="model", how="inner")

comparison["clean_acc_delta"] = comparison["clean_acc_after"] - comparison["clean_acc_before"]
comparison["clean_f1_delta"] = comparison["clean_f1_after"] - comparison["clean_f1_before"]
comparison["adv_acc_delta"] = comparison["adv_acc_after"] - comparison["adv_acc_before"]
comparison["adv_f1_delta"] = comparison["adv_f1_after"] - comparison["adv_f1_before"]

# 4) Display: sort by adversarial F1 improvement (main robustness signal)
cols = [
    "model",
    "clean_acc_before", "clean_f1_before", "adv_acc_before", "adv_f1_before",
    "clean_acc_after", "clean_f1_after", "adv_acc_after", "adv_f1_after",
    "clean_acc_delta", "clean_f1_delta", "adv_acc_delta", "adv_f1_delta",
]
display(comparison[cols].sort_values("adv_f1_delta", ascending=False).round(4))


“Before” corresponds to the baseline model trained on clean data only (Section 4.8.3), while “after” corresponds to adversarially augmented training (Section 4.8.4). “Clean” metrics are computed on the original test set, and “adv” metrics on adversarial versions of the same test samples. Augmentation improves adversarial performance for LogReg and LinearSVM (positive adv_*_delta), with the largest robustness gain for LinearSVM, but at the cost of reduced clean performance. LogReg shows the best trade-off, improving robustness while keeping (and slightly improving) clean-test scores, whereas RandomForest slightly degrades on both clean and adversarial tests.